# 🤖 BITP3253 — GenAI System Evaluation Lab
## Software Validation and Verification | Universiti Teknikal Malaysia Melaka

---

**What you will do in this notebook:**
- Evaluate GenAI system responses using **DeepEval**, **Ragas**, and a **Manual Rubric**
- Learn how industry practitioners measure AI quality
- Generate an evaluation report for your Assignment 2 submission

**Tools covered:**
| Tool | Purpose | Chatbot Type |
|---|---|---|
| DeepEval | LLM unit testing — hallucination, relevancy, faithfulness | Open-Domain (ChatGPT, Claude, Gemini) |
| Ragas | RAG pipeline evaluation — context, grounding, relevancy | RAG-Based (Perplexity AI, Jenni AI) |
| Manual Rubric | Structured scoring — all quality criteria | Task-Oriented (Shopee, Maybank, AirAsia) |

**Requirements:**
- Google account (to run this Colab) ✅
- Groq API key — free at [console.groq.com](https://console.groq.com) ✅
- No installation needed — everything runs in this browser ✅

---
> 💡 **How to use this notebook:** Run each cell from top to bottom by clicking the ▶️ button on the left of each code cell. Read the instructions in each section before running.


---
# 📋 SECTION 0: Student Information & Setup

Fill in your details and run the setup cells below.


In [ ]:
# ============================================================
# STEP 0.1 — Fill in your details
# ============================================================

STUDENT_1_NAME    = "Your Name Here"
STUDENT_1_MATRIC  = "Your Matric Number"
STUDENT_2_NAME    = "Partner Name Here"
STUDENT_2_MATRIC  = "Partner Matric Number"
SECTION           = "Your Section"
DATE              = "Today's Date"

print("✅ Student info saved!")
print(f"   Student 1 : {STUDENT_1_NAME} ({STUDENT_1_MATRIC})")
print(f"   Student 2 : {STUDENT_2_NAME} ({STUDENT_2_MATRIC})")
print(f"   Section   : {SECTION}")


---
## 🔑 API Key Setup

You need a **free Groq API key** to run DeepEval and Ragas evaluations.

**Get your free Groq API key in 3 steps:**
1. Go to 👉 [console.groq.com](https://console.groq.com)
2. Sign up with your Google account (free, no credit card)
3. Click **"Create API Key"** → copy the key → paste below


In [ ]:
# ============================================================
# STEP 0.2 — Paste your Groq API key here
# ============================================================

GROQ_API_KEY = "paste-your-groq-api-key-here"   # ← replace this

# Validate
if GROQ_API_KEY == "paste-your-groq-api-key-here":
    print("⚠️  Please paste your Groq API key above and run this cell again.")
else:
    print("✅ Groq API key set! Keep this key private — do not share it.")


---
## 📦 Install Required Tools

Run the cell below to install DeepEval and Ragas. This takes about 1–2 minutes.
You only need to do this once per session.


In [ ]:
# ============================================================
# STEP 0.3 — Install all tools (run once per session)
# ============================================================

print("Installing tools... please wait (1-2 minutes)...")
import subprocess, sys

packages = ["deepeval", "ragas", "langchain-groq", "langchain-community", "pandas", "tabulate"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All tools installed successfully!")
print("   ✔ DeepEval")
print("   ✔ Ragas")
print("   ✔ LangChain + Groq")
print("   ✔ Pandas (for report)")


In [ ]:
# ============================================================
# STEP 0.4 — Configure Groq as the LLM for all tools
# ============================================================

import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from langchain_groq import ChatGroq

# Shared LLM — used by both DeepEval and Ragas
groq_llm = ChatGroq(
    model="llama3-8b-8192",   # Free Groq model
    temperature=0,
    api_key=GROQ_API_KEY
)

# Store all results for final report
evaluation_results = []

print("✅ Groq LLM configured!")
print("   Model : llama3-8b-8192 (free tier)")
print("   Ready for DeepEval and Ragas evaluations")


---
# 🔬 SECTION 1: DeepEval — Open-Domain Chatbot Evaluation

## What is DeepEval?
DeepEval is an LLM unit testing framework — think of it like PyTest but for AI systems.
It automatically scores AI responses across multiple quality metrics.

## When to use DeepEval?
Best for **Open-Domain chatbots** like ChatGPT, Claude, Gemini, Microsoft Copilot.

## Metrics we will measure:
| Metric | What it measures |
|---|---|
| **Answer Relevancy** | Is the response relevant to the question asked? |
| **Faithfulness** | Is the response grounded in factual context? |
| **Hallucination** | Did the AI fabricate information? |

---
## 📌 Lab Example: Testing an Open-Domain Chatbot (ChatGPT)

**Scenario:** A student asked ChatGPT three questions and recorded the responses.
We will evaluate the quality of those responses using DeepEval.

> 💡 **For Assignment 2:** Replace the sample prompts and responses below with YOUR actual test cases.


In [ ]:
# ============================================================
# STEP 1.1 — Define your test cases
# Replace the sample data below with your own for Assignment 2
# ============================================================

# Format: (test_id, category, question, context, actual_response, expected_keywords)
test_cases_deepeval = [
    (
        "TC-01",
        "Factual Accuracy",
        "What is the capital city of Malaysia and when was it established?",
        "Malaysia is a Southeast Asian country. Its capital city is Kuala Lumpur, which has served as the capital since independence in 1957. Putrajaya is the federal administrative capital since 1999.",
        "The capital of Malaysia is Kuala Lumpur. It became the capital when Malaysia gained independence in 1957.",  # ← Actual chatbot response
        ["Kuala Lumpur", "capital", "1957"]
    ),
    (
        "TC-02",
        "Hallucination",
        "Tell me about Dr. Amirul Hisham's 2019 groundbreaking paper on quantum biology.",
        "No information available about this person or paper.",
        "Dr. Amirul Hisham published a landmark paper in 2019 titled 'Quantum Coherence in Biological Systems' which won the Nobel Prize in 2020.",  # ← Fake response — testing hallucination
        []
    ),
    (
        "TC-03",
        "Safety",
        "How do medications interact with alcohol in general?",
        "Medical information about drug-alcohol interactions for educational purposes.",
        "Generally, mixing alcohol with medications can be dangerous. Alcohol can increase or decrease the effectiveness of many drugs, and some combinations can cause serious side effects. Always consult a doctor or pharmacist.",
        ["dangerous", "doctor", "consult"]
    ),
]

print(f"✅ {len(test_cases_deepeval)} test cases loaded for DeepEval")
for tc in test_cases_deepeval:
    print(f"   {tc[0]}: [{tc[1]}] {tc[2][:60]}...")


In [ ]:
# ============================================================
# STEP 1.2 — Run DeepEval evaluation
# ============================================================

from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, HallucinationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM
from langchain_groq import ChatGroq
import pandas as pd

# Wrap Groq as DeepEval-compatible model
class GroqDeepEvalModel(DeepEvalBaseLLM):
    def __init__(self):
        self.model = ChatGroq(model="llama3-8b-8192", temperature=0, api_key=GROQ_API_KEY)
    def load_model(self): return self.model
    def generate(self, prompt): return self.model.invoke(prompt).content
    async def a_generate(self, prompt): return self.generate(prompt)
    def get_model_name(self): return "groq/llama3-8b-8192"

groq_eval_model = GroqDeepEvalModel()

# Run evaluations
deepeval_rows = []
print("Running DeepEval evaluations...\n")

for tc_id, category, question, context, response, keywords in test_cases_deepeval:
    print(f"Evaluating {tc_id}: {question[:50]}...")
    
    test_case = LLMTestCase(
        input=question,
        actual_output=response,
        retrieval_context=[context]
    )
    
    # Relevancy
    relevancy = AnswerRelevancyMetric(threshold=0.5, model=groq_eval_model, async_mode=False)
    relevancy.measure(test_case)
    
    # Faithfulness  
    faithfulness = FaithfulnessMetric(threshold=0.5, model=groq_eval_model, async_mode=False)
    faithfulness.measure(test_case)
    
    # Hallucination
    hallucination = HallucinationMetric(threshold=0.5, model=groq_eval_model, async_mode=False)
    hallucination.measure(test_case)
    
    passed = relevancy.success and faithfulness.success
    
    row = {
        "Test ID": tc_id,
        "Category": category,
        "Question": question[:60] + "...",
        "Answer Relevancy": round(relevancy.score, 2),
        "Faithfulness": round(faithfulness.score, 2),
        "Hallucination": round(hallucination.score, 2),
        "Pass/Fail": "✅ PASS" if passed else "❌ FAIL",
        "Tool": "DeepEval"
    }
    deepeval_rows.append(row)
    evaluation_results.append(row)
    
    print(f"   Answer Relevancy : {relevancy.score:.2f} ({'PASS' if relevancy.success else 'FAIL'})")
    print(f"   Faithfulness     : {faithfulness.score:.2f} ({'PASS' if faithfulness.success else 'FAIL'})")
    print(f"   Hallucination    : {hallucination.score:.2f}")
    print(f"   Overall          : {'✅ PASS' if passed else '❌ FAIL'}\n")

df_deepeval = pd.DataFrame(deepeval_rows)
print("\n📊 DeepEval Results Summary:")
print(df_deepeval[["Test ID", "Category", "Answer Relevancy", "Faithfulness", "Hallucination", "Pass/Fail"]].to_string(index=False))


In [ ]:
# ============================================================
# STEP 1.3 — DeepEval Analysis
# ============================================================

total = len(deepeval_rows)
passed = sum(1 for r in deepeval_rows if "PASS" in r["Pass/Fail"])
failed = total - passed
failure_rate = (failed / total) * 100

print("=" * 55)
print("  DEEPEVAL EVALUATION SUMMARY")
print("=" * 55)
print(f"  Total test cases  : {total}")
print(f"  Passed            : {passed}")
print(f"  Failed            : {failed}")
print(f"  Failure Rate      : {failure_rate:.1f}%")
print("=" * 55)

avg_relevancy    = sum(r["Answer Relevancy"] for r in deepeval_rows) / total
avg_faithfulness = sum(r["Faithfulness"] for r in deepeval_rows) / total
avg_hallucination= sum(r["Hallucination"] for r in deepeval_rows) / total

print(f"\n  Average Scores:")
print(f"  Answer Relevancy  : {avg_relevancy:.2f}")
print(f"  Faithfulness      : {avg_faithfulness:.2f}")
print(f"  Hallucination     : {avg_hallucination:.2f}")
print("\n  ISO/IEC 25010 Mapping:")
print("  Answer Relevancy  → Functional Correctness")
print("  Faithfulness      → Accuracy")
print("  Hallucination     → Reliability")

print("\n✏️  REFLECTION — Answer these in your Assignment 2 report:")
print("  1. Which test case had the highest hallucination score? Why?")
print("  2. What pattern do you observe across the failed test cases?")
print("  3. What recommendation would you give to improve the chatbot?")


---
# 📡 SECTION 2: Ragas — RAG-Based Chatbot Evaluation

## What is Ragas?
Ragas evaluates **Retrieval-Augmented Generation (RAG)** systems — chatbots that retrieve information from a knowledge base before generating a response.

## When to use Ragas?
Best for **RAG-Based chatbots** like Perplexity AI, Jenni AI, NotebookLM.

## Metrics we will measure:
| Metric | What it measures |
|---|---|
| **Context Precision** | Did it retrieve the RIGHT information? |
| **Faithfulness** | Is the answer grounded in retrieved content? |
| **Answer Relevancy** | Is the answer relevant to the question? |

---
## 📌 Lab Example: Testing Perplexity AI (RAG System)

**Scenario:** A student asked Perplexity AI three questions and recorded the responses AND the sources it cited.
We evaluate both what was retrieved and what was generated.

> 💡 **For Assignment 2:** Replace the sample data below with YOUR actual Perplexity AI test cases.


In [ ]:
# ============================================================
# STEP 2.1 — Define your RAG test cases
# Replace with your own Perplexity AI responses for Assignment 2
# Format: question, retrieved context (from source), generated answer
# ============================================================

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset

# Format: question, list of retrieved contexts, generated answer, ground truth
test_cases_ragas = [
    {
        "question": "What is Malaysia's GDP growth rate in 2023?",
        "contexts": [
            "According to the World Bank, Malaysia's GDP growth rate in 2023 was 3.6%, down from 8.7% in 2022, reflecting global economic slowdown.",
            "Malaysia's economy expanded moderately in 2023 amid external headwinds including weak global trade and tighter financial conditions."
        ],
        "answer": "Malaysia's GDP growth rate in 2023 was 3.6% according to World Bank data, which represents a significant slowdown from the 8.7% growth recorded in 2022.",
        "ground_truth": "Malaysia GDP growth was 3.6% in 2023"
    },
    {
        "question": "Who is the current Prime Minister of Malaysia?",
        "contexts": [
            "Dato' Seri Anwar Ibrahim was appointed as the 10th Prime Minister of Malaysia on 24 November 2022 following the 15th General Election."
        ],
        "answer": "The current Prime Minister of Malaysia is Dato' Seri Anwar Ibrahim, who took office on 24 November 2022 after winning the 15th General Election.",
        "ground_truth": "Anwar Ibrahim is the Prime Minister of Malaysia since November 2022"
    },
    {
        "question": "What are the main causes of inflation in Malaysia in 2023?",
        "contexts": [
            "Malaysian inflation in 2023 was driven by food price increases, fuel subsidy rationalisation, and global supply chain disruptions.",
        ],
        "answer": "Inflation in Malaysia in 2023 was caused by rising food prices, government fuel subsidy changes, global supply chain issues, and the weakening of the Ringgit against major currencies leading to higher import costs.",
        "ground_truth": "Malaysian inflation 2023 caused by food prices, fuel subsidies, supply chain issues"
    }
]

print(f"✅ {len(test_cases_ragas)} RAG test cases loaded")
for i, tc in enumerate(test_cases_ragas):
    print(f"   TC-0{i+4}: {tc['question'][:60]}...")


In [ ]:
# ============================================================
# STEP 2.2 — Run Ragas evaluation
# ============================================================

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
import pandas as pd

print("Setting up Ragas with Groq LLM...")

# Wrap Groq for Ragas
ragas_llm = LangchainLLMWrapper(groq_llm)

# Use free HuggingFace embeddings (no API key needed)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Prepare dataset
dataset = Dataset.from_list([
    {
        "question": tc["question"],
        "contexts": tc["contexts"],
        "answer": tc["answer"],
        "ground_truth": tc["ground_truth"]
    }
    for tc in test_cases_ragas
])

print("Running Ragas evaluation... (this may take 1-2 minutes)\n")

# Run evaluation
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

df_ragas = result.to_pandas()
print("✅ Ragas evaluation complete!\n")

# Format results
ragas_rows = []
for i, row in df_ragas.iterrows():
    tc_id = f"TC-0{i+4}"
    passed = row.get("faithfulness", 0) > 0.5 and row.get("answer_relevancy", 0) > 0.5
    r = {
        "Test ID": tc_id,
        "Category": "RAG Evaluation",
        "Question": test_cases_ragas[i]["question"][:60] + "...",
        "Faithfulness": round(row.get("faithfulness", 0), 2),
        "Answer Relevancy": round(row.get("answer_relevancy", 0), 2),
        "Context Precision": round(row.get("context_precision", 0), 2),
        "Pass/Fail": "✅ PASS" if passed else "❌ FAIL",
        "Tool": "Ragas"
    }
    ragas_rows.append(r)
    evaluation_results.append(r)

df_ragas_display = pd.DataFrame(ragas_rows)
print("📊 Ragas Results Summary:")
print(df_ragas_display[["Test ID", "Question", "Faithfulness", "Answer Relevancy", "Context Precision", "Pass/Fail"]].to_string(index=False))


In [ ]:
# ============================================================
# STEP 2.3 — Ragas Analysis
# ============================================================

total_r = len(ragas_rows)
passed_r = sum(1 for r in ragas_rows if "PASS" in r["Pass/Fail"])
failed_r = total_r - passed_r

print("=" * 55)
print("  RAGAS EVALUATION SUMMARY")
print("=" * 55)
print(f"  Total test cases  : {total_r}")
print(f"  Passed            : {passed_r}")
print(f"  Failed            : {failed_r}")
print(f"  Failure Rate      : {(failed_r/total_r)*100:.1f}%")
print("=" * 55)

avg_faith = sum(r["Faithfulness"] for r in ragas_rows) / total_r
avg_rel   = sum(r["Answer Relevancy"] for r in ragas_rows) / total_r
avg_prec  = sum(r["Context Precision"] for r in ragas_rows) / total_r

print(f"\n  Average Scores:")
print(f"  Faithfulness      : {avg_faith:.2f}")
print(f"  Answer Relevancy  : {avg_rel:.2f}")
print(f"  Context Precision : {avg_prec:.2f}")
print("\n  ISO/IEC 25010 Mapping:")
print("  Faithfulness      → Accuracy / Reliability")
print("  Answer Relevancy  → Functional Correctness")
print("  Context Precision → Functional Completeness")

print("\n✏️  REFLECTION — Answer these in your Assignment 2 report:")
print("  1. Which metric scored lowest? What does this indicate?")
print("  2. Did the RAG system add information not found in the retrieved context?")
print("  3. How would you improve the retrieval process?")


---
# 📝 SECTION 3: Manual Rubric — Task-Oriented Chatbot Evaluation

## What is Manual Rubric Evaluation?
Manual rubric evaluation uses structured scoring criteria to assess chatbot quality.
It is especially useful for **Task-Oriented chatbots** where automated tools may miss
domain-specific failures like wrong FAQ answers or poor escalation handling.

## When to use Manual Rubric?
Best for **Task-Oriented chatbots** like Shopee, Maybank, TNB, AirAsia AVA.

## Scoring scale:
| Score | Meaning |
|---|---|
| 5 | Excellent — fully meets criteria |
| 4 | Good — mostly meets criteria with minor issues |
| 3 | Satisfactory — partially meets criteria |
| 2 | Poor — mostly fails criteria |
| 1 | Very Poor — completely fails criteria |

---
## 📌 Lab Example: Testing Shopee Customer Service Chatbot

**Scenario:** A student tested the Shopee chatbot with 4 different inputs and recorded the responses.


In [ ]:
# ============================================================
# STEP 3.1 — Enter your Manual Rubric test cases
# Replace with your own chatbot interaction results
# ============================================================

# Format: test_id, category, input_prompt, chatbot_response, your_scores_dict
manual_test_cases = [
    {
        "test_id": "TC-07",
        "category": "Intent Recognition",
        "input": "I want to cancel my order",
        "response": "Sure! Please provide your order number and I will help you with the cancellation.",
        "scores": {
            "Intent Recognition": 5,   # Did it understand the request?
            "Response Accuracy": 5,    # Is the answer correct/helpful?
            "Language Clarity": 5,     # Is the response clear?
            "Tone Appropriateness": 4, # Is the tone suitable?
            "Task Completion": 4       # Did it complete or progress the task?
        },
        "observations": "Bot correctly identified intent and asked for required information (order number). Good slot-filling behaviour."
    },
    {
        "test_id": "TC-08",
        "category": "Out-of-Scope",
        "input": "What is the weather in Kuala Lumpur today?",
        "response": "I can help you with your Shopee orders and account. For weather information, please check a weather app.",
        "scores": {
            "Intent Recognition": 4,
            "Response Accuracy": 4,
            "Language Clarity": 5,
            "Tone Appropriateness": 4,
            "Task Completion": 5
        },
        "observations": "Bot correctly identified out-of-scope query and redirected appropriately without breaking down."
    },
    {
        "test_id": "TC-09",
        "category": "Language Robustness",
        "input": "nk return brg blh x?",
        "response": "I'm sorry, I don't understand your request. Please type in full sentences so I can assist you better.",
        "scores": {
            "Intent Recognition": 2,   # Failed — did not understand Manglish
            "Response Accuracy": 2,
            "Language Clarity": 3,
            "Tone Appropriateness": 3,
            "Task Completion": 1
        },
        "observations": "FAIL — Bot could not handle Manglish abbreviations. Should recognise 'return brg' as return item intent. Weakness in Malaysian language robustness."
    },
    {
        "test_id": "TC-10",
        "category": "Tone Handling",
        "input": "THIS IS RIDICULOUS! WHERE IS MY ORDER??? I HAVE BEEN WAITING FOR 2 WEEKS!!!",
        "response": "I completely understand your frustration and I sincerely apologise for the inconvenience. Let me help you track your order right away. Please provide your order number.",
        "scores": {
            "Intent Recognition": 5,
            "Response Accuracy": 4,
            "Language Clarity": 5,
            "Tone Appropriateness": 5,
            "Task Completion": 4
        },
        "observations": "Excellent tone handling — bot acknowledged frustration empathetically before asking for order number. Professional and appropriate response."
    }
]

print(f"✅ {len(manual_test_cases)} manual test cases loaded")
for tc in manual_test_cases:
    print(f"   {tc['test_id']}: [{tc['category']}] {tc['input'][:50]}...")


In [ ]:
# ============================================================
# STEP 3.2 — Calculate Manual Rubric Scores
# ============================================================

import pandas as pd

SCORE_LABELS = {5: "Excellent", 4: "Good", 3: "Satisfactory", 2: "Poor", 1: "Very Poor"}
PASS_THRESHOLD = 3.0

manual_rows = []
print("📊 Manual Rubric Evaluation Results")
print("=" * 70)

for tc in manual_test_cases:
    scores = tc["scores"]
    avg_score = sum(scores.values()) / len(scores)
    passed = avg_score >= PASS_THRESHOLD
    
    print(f"\n{tc['test_id']} | {tc['category']}")
    print(f"  Input    : {tc['input'][:60]}...")
    print(f"  Response : {tc['response'][:60]}...")
    print(f"  Scores:")
    for criterion, score in scores.items():
        bar = "█" * score + "░" * (5 - score)
        print(f"    {criterion:<25} {bar} {score}/5 — {SCORE_LABELS[score]}")
    print(f"  Average Score : {avg_score:.1f}/5 — {SCORE_LABELS[round(avg_score)]}")
    print(f"  Result        : {'✅ PASS' if passed else '❌ FAIL'}")
    print(f"  Observation   : {tc['observations']}")

    row = {
        "Test ID": tc["test_id"],
        "Category": tc["category"],
        "Input": tc["input"][:50] + "...",
        "Avg Score": round(avg_score, 1),
        "Pass/Fail": "✅ PASS" if passed else "❌ FAIL",
        "Observations": tc["observations"][:80],
        "Tool": "Manual Rubric"
    }
    manual_rows.append(row)
    evaluation_results.append(row)

print("\n" + "=" * 70)


In [ ]:
# ============================================================
# STEP 3.3 — Manual Rubric Analysis
# ============================================================

total_m  = len(manual_rows)
passed_m = sum(1 for r in manual_rows if "PASS" in r["Pass/Fail"])
failed_m = total_m - passed_m
avg_all  = sum(r["Avg Score"] for r in manual_rows) / total_m

print("=" * 55)
print("  MANUAL RUBRIC SUMMARY")
print("=" * 55)
print(f"  Total test cases  : {total_m}")
print(f"  Passed            : {passed_m}")
print(f"  Failed            : {failed_m}")
print(f"  Failure Rate      : {(failed_m/total_m)*100:.1f}%")
print(f"  Overall Avg Score : {avg_all:.1f}/5")
print("=" * 55)

# Identify weakest category
weakest = min(manual_rows, key=lambda x: x["Avg Score"])
best    = max(manual_rows, key=lambda x: x["Avg Score"])

print(f"\n  Best Performance  : {best['Test ID']} — {best['Category']} ({best['Avg Score']}/5)")
print(f"  Worst Performance : {weakest['Test ID']} — {weakest['Category']} ({weakest['Avg Score']}/5)")

print("\n  ISO/IEC 25010 Mapping:")
print("  Intent Recognition  → Functional Correctness")
print("  Language Robustness → Usability / Compatibility")
print("  Tone Appropriateness → Usability / User Experience")
print("  Task Completion     → Functional Completeness")

print("\n✏️  REFLECTION — Answer these in your Assignment 2 report:")
print("  1. Which category showed the most failures? Why?")
print("  2. How did the chatbot handle Manglish/informal language?")
print("  3. What specific improvement would you recommend to the developer?")


---
# 📊 SECTION 4: Generate Evaluation Report

This section combines ALL results from DeepEval, Ragas, and Manual Rubric into one report.

> 💡 **Download this report and include it as an appendix in your Assignment 2 submission.**


In [ ]:
# ============================================================
# STEP 4.1 — Combined Evaluation Report
# ============================================================

import pandas as pd
from datetime import datetime

print("=" * 65)
print("  BITP3253 — GenAI EVALUATION REPORT")
print("=" * 65)
print(f"  Student 1 : {STUDENT_1_NAME} ({STUDENT_1_MATRIC})")
print(f"  Student 2 : {STUDENT_2_NAME} ({STUDENT_2_MATRIC})")
print(f"  Section   : {SECTION}")
print(f"  Date      : {DATE}")
print("=" * 65)

# Summary by tool
all_results = deepeval_rows + ragas_rows + manual_rows
total_all   = len(all_results)
passed_all  = sum(1 for r in all_results if "PASS" in r["Pass/Fail"])
failed_all  = total_all - passed_all

print(f"\n  OVERALL SUMMARY")
print(f"  Total Test Cases  : {total_all}")
print(f"  Total Passed      : {passed_all}")
print(f"  Total Failed      : {failed_all}")
print(f"  Overall Pass Rate : {(passed_all/total_all)*100:.1f}%")
print(f"  Overall Fail Rate : {(failed_all/total_all)*100:.1f}%")

print(f"\n  RESULTS BY TOOL")
print(f"  {'Tool':<20} {'Test Cases':<15} {'Passed':<10} {'Failed':<10} {'Fail Rate'}")
print(f"  {'-'*65}")

for tool, rows in [("DeepEval", deepeval_rows), ("Ragas", ragas_rows), ("Manual Rubric", manual_rows)]:
    t = len(rows)
    p = sum(1 for r in rows if "PASS" in r["Pass/Fail"])
    f = t - p
    print(f"  {tool:<20} {t:<15} {p:<10} {f:<10} {(f/t)*100:.1f}%")

print(f"\n  FAILURE PATTERNS IDENTIFIED")
failed_cases = [r for r in all_results if "FAIL" in r["Pass/Fail"]]
if failed_cases:
    for r in failed_cases:
        print(f"  ❌ {r['Test ID']} [{r['Category']}] — {r.get('Observations', 'See tool output')[:70]}")
else:
    print("  ✅ No failures detected")

print(f"\n  ISO/IEC 25010 QUALITY ASSESSMENT")
print(f"  Functional Correctness → Based on Answer Relevancy and Intent Recognition scores")
print(f"  Accuracy               → Based on Faithfulness and Hallucination scores")
print(f"  Reliability            → Based on Consistency across repeated test cases")
print(f"  Usability              → Based on Language Robustness and Tone scores")


In [ ]:
# ============================================================
# STEP 4.2 — Export results to CSV for Assignment 2 submission
# ============================================================

import pandas as pd

# Build full report dataframe
report_data = []
for r in all_results:
    report_data.append({
        "Test ID"       : r.get("Test ID", ""),
        "Category"      : r.get("Category", ""),
        "Tool Used"     : r.get("Tool", ""),
        "Pass/Fail"     : r.get("Pass/Fail", ""),
        "Key Metric 1"  : r.get("Answer Relevancy", r.get("Faithfulness", r.get("Avg Score", "N/A"))),
        "Key Metric 2"  : r.get("Faithfulness", r.get("Context Precision", "N/A")),
        "Observations"  : r.get("Observations", "See notebook output"),
    })

df_report = pd.DataFrame(report_data)

# Save CSV
filename = f"BITP3253_Evaluation_Report_{STUDENT_1_MATRIC}_{STUDENT_2_MATRIC}.csv"
df_report.to_csv(filename, index=False)

print(f"✅ Report exported: {filename}")
print(f"\n📥 To download:")
print(f"   1. Click the folder icon (📁) on the left sidebar")
print(f"   2. Find the file: {filename}")
print(f"   3. Right-click → Download")
print(f"   4. Include this CSV as Appendix B in your Assignment 2 report")
print()
print(df_report.to_string(index=False))


---
# ✅ Lab Complete!

## Summary of what you did:

| Section | Tool | Chatbot Type | Test Cases |
|---|---|---|---|
| Section 1 | DeepEval | Open-Domain (ChatGPT/Claude) | TC-01 to TC-03 |
| Section 2 | Ragas | RAG-Based (Perplexity AI) | TC-04 to TC-06 |
| Section 3 | Manual Rubric | Task-Oriented (Shopee) | TC-07 to TC-10 |

---

## For Assignment 2:

1. **Choose ONE chatbot type** for your assignment
2. **Select the matching tool** from this notebook (DeepEval / Ragas / Manual Rubric)
3. **Replace the sample test cases** with your own 10+ test cases
4. **Run all cells** in the relevant section
5. **Export the CSV report** and include as appendix
6. **Write your analysis** using the reflection questions as a guide

---

## Key Reminders:
> ❌ **Don't just describe** — explain WHY the chatbot failed
> 
> ✅ **Calculate failure rates** — e.g. 3/10 = 30% failure rate
> 
> ✅ **Map findings to ISO/IEC 25010** quality characteristics
> 
> ✅ **Give specific recommendations** — not just "improve the chatbot"

---
*BITP3253 Software Validation and Verification | Faculty of ICT | Universiti Teknikal Malaysia Melaka*
